# Data Preprocessing

This notebook loads the heart disease dataset, performs basic preprocessing, and saves the cleaned data into the processed data folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()

    for path in [current, *current.parents]:
        # Use the presence of the data folder to detect the project root.
        if (path / 'data').exists():
            return path

    raise FileNotFoundError('Project root not found.')


# Build the key paths used in this notebook.
BASE_DIR = find_project_root(Path.cwd())
RAW_DATA_PATH = BASE_DIR / 'data' / 'raw' / 'dataset.csv'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH = PROCESSED_DIR / 'heart_disease_processed.csv'

BASE_DIR, RAW_DATA_PATH, PROCESSED_DATA_PATH

(WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/data/raw/dataset.csv'),
 WindowsPath('D:/PROJECT OF DATA SCIENCE/Heart Disease Health Indicators/data/processed/heart_disease_processed.csv'))

In [3]:
# Load the raw dataset for preprocessing.
data = pd.read_csv(RAW_DATA_PATH)
df = data.copy()
df.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [4]:
print(f'Shape before preprocessing: {df.shape}')
print('\nMissing values per column:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

Shape before preprocessing: (253680, 22)

Missing values per column:
HeartDiseaseorAttack    0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
Diabetes                0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Duplicate rows: 23899


In [5]:
# Define the target and columns that should stay as binary integers.
target_column = 'HeartDiseaseorAttack'

binary_columns = [
    'HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'Smoker',
    'Stroke', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump',
    'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex'
]


In [6]:
df[[col for col in binary_columns if col in df.columns]].head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,Smoker,Stroke,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,DiffWalk,Sex
0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0
3,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0


In [7]:
# Remove duplicate rows to avoid training bias from repeated samples.
df = df.drop_duplicates().copy()

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
# Fill any missing numeric values with the median of each column.
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_columns] = df[numeric_columns].apply(lambda col: col.fillna(col.median()))

In [10]:
# Convert binary columns and the target column to integer type.
for column in binary_columns:
    if column in df.columns:
        df[column] = df[column].round().astype(int)

if target_column in df.columns:
    df[target_column] = df[target_column].astype(int)

df.head()

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,1,1,1,40.0,1,0,0.0,0,0,...,1,0,5.0,18.0,15.0,1,0,9.0,4.0,3.0
1,0,0,0,0,25.0,1,0,0.0,1,0,...,0,1,3.0,0.0,0.0,0,0,7.0,6.0,1.0
2,0,1,1,1,28.0,0,0,0.0,0,1,...,1,1,5.0,30.0,30.0,1,0,9.0,4.0,8.0
3,0,1,0,1,27.0,0,0,0.0,1,1,...,1,0,2.0,0.0,0.0,0,0,11.0,3.0,6.0
4,0,1,1,1,24.0,0,0,0.0,1,1,...,1,0,2.0,3.0,0.0,0,0,11.0,5.0,4.0


In [11]:
print(f'Shape after preprocessing: {df.shape}')
print('\nData types:')
print(df.dtypes)

Shape after preprocessing: (229781, 22)

Data types:
HeartDiseaseorAttack      int64
HighBP                    int64
HighChol                  int64
CholCheck                 int64
BMI                     float64
Smoker                    int64
Stroke                    int64
Diabetes                float64
PhysActivity              int64
Fruits                    int64
Veggies                   int64
HvyAlcoholConsump         int64
AnyHealthcare             int64
NoDocbcCost               int64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                  int64
Sex                       int64
Age                     float64
Education               float64
Income                  float64
dtype: object


In [12]:
# Save the cleaned dataset for the modeling notebooks.
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f'Processed data saved to: {PROCESSED_DATA_PATH}')

Processed data saved to: D:\PROJECT OF DATA SCIENCE\Heart Disease Health Indicators\data\processed\heart_disease_processed.csv
